In [1]:
import yt

In [2]:
#PATH TO SIMULATION
filepath = "C:/Users/jonat/source/repos/UMD/GEMS/ricotti/output_00273/info_00273.txt"

In [3]:
epf = [
    ("particle_family", "b"),
    ("particle_tag", "b"),
    ("particle_birth_epoch", "d"),
    ("particle_metallicity", "d"),
]

ds = yt.load(filepath, extra_particle_fields = epf)
ad = ds.all_data()

yt : [INFO     ] 2026-03-23 17:33:00,076 Parameters: current_time              = 0.3604448649237178 Gyr
yt : [INFO     ] 2026-03-23 17:33:00,078 Parameters: domain_dimensions         = [64 64 64]
yt : [INFO     ] 2026-03-23 17:33:00,079 Parameters: domain_left_edge          = [0. 0. 0.]
yt : [INFO     ] 2026-03-23 17:33:00,081 Parameters: domain_right_edge         = [1. 1. 1.]
yt : [INFO     ] 2026-03-23 17:33:00,082 Parameters: cosmological_simulation   = True
yt : [INFO     ] 2026-03-23 17:33:00,083 Parameters: current_redshift          = 12.171087046255657
yt : [INFO     ] 2026-03-23 17:33:00,084 Parameters: omega_lambda              = 0.685000002384186
yt : [INFO     ] 2026-03-23 17:33:00,085 Parameters: omega_matter              = 0.314999997615814
yt : [INFO     ] 2026-03-23 17:33:00,086 Parameters: omega_radiation           = 0.0
yt : [INFO     ] 2026-03-23 17:33:00,090 Parameters: hubble_constant           = 0.674000015258789
yt : [WARNING  ] 2026-03-23 17:33:00,951 This output

In [4]:
import numpy as np

Calculate stellar age (Fred Garcia's [TIRAMISU](https://github.com/fred144/tiramisu/blob/main/tools/cosmo.py))

In [5]:
cgs_yr = 3.1556926e7  # 1yr (in s)
cgs_pc = 3.08567758e18  # pc (in cm)
h_0 = ds.hubble_constant * 100 # hubble parameter (km/s/Mpc)
h_0_invsec = h_0 * 1e5 / (1e6 * cgs_pc)  # hubble constant h [km/s Mpc-1]->[1/sec]
h_0inv_yr = 1 / h_0_invsec / cgs_yr  # 1/h_0 [yr]
star_age_myr = np.array(ad["star", "particle_birth_epoch"]) * h_0inv_yr / 1e6 + 13700

yt : [INFO     ] 2026-03-23 17:33:05,333 Identified   384/  384 intersecting domains (  385 through hilbert key indexing)


In [6]:
time = ds.current_time.in_units("Myr").value
stellar_ages = time - star_age_myr

In [7]:
x_pc = ad["star", "particle_position_x"].in_units("pc")
y_pc = ad["star", "particle_position_y"].in_units("pc")
z_pc = ad["star", "particle_position_z"].in_units("pc")

center_pc = (np.mean(x_pc), np.mean(y_pc), np.mean(z_pc))

x2 = x_pc - center_pc[0]
y2 = y_pc - center_pc[1]
z2 = y_pc - center_pc[2]

xyz_centered = np.array([(x2[i], y2[i], z2[i]) for i in range(len(x2))])

In [8]:
import convert_text

Create silmaril input file (Kevin Li)

In [10]:
convert_text.convert_to_text_from_ds(ds)

'output.txt'

Calculate line emissions (Braden Nowicki, [Merlin](https://github.com/BradenN6/Merlin))

In [14]:
import os
import copy

import numpy as np
import yt
from yt.frontends.ramses.field_handlers import RTFieldFileHandler
import matplotlib.pyplot as plt
import scipy

from merlin_spectra.emission import EmissionLineInterpolator
from merlin_spectra import galaxy_visualization

import silmaril

In [12]:
lines=["H1_6562.80A","O1_1304.86A","O1_6300.30A","O2_3728.80A","O2_3726.10A",
       "O3_1660.81A","O3_1666.15A","O3_4363.21A","O3_4958.91A","O3_5006.84A", 
       "He2_1640.41A","C2_1335.66A","C3_1906.68A","C3_1908.73A","C4_1549.00A",
       "Mg2_2795.53A","Mg2_2802.71A","Ne3_3868.76A","Ne3_3967.47A",
       "N5_1238.82A",
       "N5_1242.80A","N4_1486.50A","N3_1749.67A","S2_6716.44A","S2_6730.82A"]

wavelengths=[6562.80, 1304.86, 6300.30, 3728.80, 3726.10, 1660.81, 1666.15,
             4363.21, 4958.91, 5006.84, 1640.41, 1335.66,
             1906.68, 1908.73, 1549.00, 2795.53, 2802.71, 3868.76,
             3967.47, 1238.82, 1242.80, 1486.50, 1749.67, 6716.44, 6730.82]

cell_fields = [
    "Density",
    "x-velocity",
    "y-velocity",
    "z-velocity",
    "Pressure",
    "Metallicity",
    "xHI",
    "xHII",
    "xHeII",
    "xHeIII",
]

epf = [
    ("particle_family", "b"),
    ("particle_tag", "b"),
    ("particle_birth_epoch", "d"),
    ("particle_metallicity", "d"),
]

In [15]:
# Ionization Parameter Field
# Based on photon densities in bins 2-4
# Don't include bin 1 -> Lyman Werner non-ionizing
def _ion_param(field, data):
    p = RTFieldFileHandler.get_rt_parameters(ds).copy()
    p.update(ds.parameters)

    cgs_c = 2.99792458e10     #light velocity

    # Convert to physical photon number density in cm^-3
    pd_2 = data['ramses-rt','Photon_density_2']*p["unit_pf"]/cgs_c
    pd_3 = data['ramses-rt','Photon_density_3']*p["unit_pf"]/cgs_c
    pd_4 = data['ramses-rt','Photon_density_4']*p["unit_pf"]/cgs_c

    photon = pd_2 + pd_3 + pd_4

    return photon/data['gas', 'number_density']


def _my_temperature(field, data):
    #y(i): abundance per hydrogen atom
    XH_RAMSES=0.76 #defined by RAMSES in cooling_module.f90
    YHE_RAMSES=0.24 #defined by RAMSES in cooling_module.f90
    mH_RAMSES=yt.YTArray(1.6600000e-24,"g") #defined by RAMSES in cooling_module.f90
    kB_RAMSES=yt.YTArray(1.3806200e-16,"erg/K") #defined by RAMSES in cooling_module.f90

    dn=data["ramses","Density"].in_cgs()
    pr=data["ramses","Pressure"].in_cgs()
    yHI=data["ramses","xHI"]
    yHII=data["ramses","xHII"]
    yHe = YHE_RAMSES*0.25/XH_RAMSES
    yHeII=data["ramses","xHeII"]*yHe
    yHeIII=data["ramses","xHeIII"]*yHe
    yH2=1.-yHI-yHII
    yel=yHII+yHeII+2*yHeIII
    mu=(yHI+yHII+2.*yH2 + 4.*yHe) / (yHI+yHII+yH2 + yHe + yel)
    return pr/dn * mu * mH_RAMSES / kB_RAMSES


#number density of hydrogen atoms
def _my_H_nuclei_density(field, data):
    dn=data["ramses","Density"].in_cgs()
    XH_RAMSES=0.76 #defined by RAMSES in cooling_module.f90
    YHE_RAMSES=0.24 #defined by RAMSES in cooling_module.f90
    mH_RAMSES=yt.YTArray(1.6600000e-24,"g") #defined by RAMSES in cooling_module.f90

    return dn*XH_RAMSES/mH_RAMSES


def _OII_ratio(field, data):
    # TODO lum or flux?
    #return data['gas', 'flux_O2_3728.80A']/data['gas', 'flux_O2_3726.10A']
    flux1 = data['gas', 'flux_O2_3728.80A']
    flux2 = data['gas', 'flux_O2_3726.10A']

    flux2 = np.where(flux2 < 1e-30, 1e-30, flux2)

    ratio = flux1 / flux2

    return ratio


def _pressure(field, data):
    if 'hydro_thermal_pressure' in dir(ds.fields.ramses): # and 
        #'Pressure' not in dir(ds.fields.ramses):
        return data['ramses', 'hydro_thermal_pressure']


def _xHI(field, data):
    if 'hydro_xHI' in dir(ds.fields.ramses): # and \
        #'xHI' not in dir(ds.fields.ramses):
        return data['ramses', 'hydro_xHI']


def _xHII(field, data):
    if 'hydro_xHII' in dir(ds.fields.ramses): # and \
        #'xHII' not in dir(ds.fields.ramses):
        return data['ramses', 'hydro_xHII']


def _xHeII(field, data):
    if 'hydro_xHeII' in dir(ds.fields.ramses): # and \
        #'xHeII' not in dir(ds.fields.ramses):
        return data['ramses', 'hydro_xHeII']


def _xHeIII(field, data):
    if 'hydro_xHeIII' in dir(ds.fields.ramses): # and \
        #'xHeIII' not in dir(ds.fields.ramses):
        return data['ramses', 'hydro_xHeIII']